# 04 — Modeling & Evaluation

Season-based splits, baselines, supervised models, and model result artifacts.

- **Target:** `points_finish`
- **Models:** Random baseline, heuristic (`first_observed_position` ≤ 10), Logistic Regression, Random Forest, LightGBM
- **Split:** Train 2023 · Validation 2024 · Test 2025 (season-based, no random row split)
- **Engine:** pandas + scikit-learn + LightGBM (Gold mart handoff)

Run after `03_gold_feature_engineering.ipynb` with the same `USE_GOOGLE_DRIVE` setting.


## Connection to grading rubric

| Rubric area | This notebook |
|-------------|----------------|
| Experimental Results & Analysis | Baselines, model metrics, confusion matrix, error analysis |
| Feature Engineering | Uses Gold mart + feature dictionary with leakage guard |
| Data Quality | Error analysis by team/circuit/season |


## Colab setup (required every session)

Identical to `00`–`03`: clone, `pip install -e .`, Drive mount, set `OPENF1_DATA_ROOT` **before** importing config.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



## Configuration


In [ ]:
MODELING_MODE = "smoke"  # "full" for official MBA season splits
CLEAR_MODEL_OUTPUTS = True

from pathlib import Path

import pandas as pd

from openf1_pipeline.config import (
    RANDOM_SEED,
    get_data_quality_reports_dir,
    get_feature_definitions_dir,
    get_gold_dir,
    get_manifests_dir,
    get_model_results_dir,
    get_output_root,
)
from openf1_pipeline.features.feature_dictionary import validate_no_leakage
from openf1_pipeline.gold.build_feature_mart import GOLD_MART_FILENAME, TARGET_COLUMN
from openf1_pipeline.modeling.baselines import (
    heuristic_position_baseline,
    random_baseline_predictions,
)
from openf1_pipeline.modeling.evaluate import (
    build_error_analysis,
    compute_classification_metrics,
    compute_confusion_matrix_table,
    save_modeling_outputs,
)
from openf1_pipeline.modeling.splits import resolve_modeling_splits
from openf1_pipeline.modeling.train import (
    build_lightgbm_pipeline,
    build_logistic_regression_pipeline,
    build_random_forest_pipeline,
    extract_feature_importance,
    get_model_feature_columns,
    prepare_model_matrix,
    train_models,
)
from openf1_pipeline.utils.cleanup import clean_model_outputs
from openf1_pipeline.utils.io import read_parquet_if_exists

GOLD_DIR = get_gold_dir()
FEATURE_DEFINITIONS_DIR = get_feature_definitions_dir()
MODEL_RESULTS_DIR = get_model_results_dir()
MANIFESTS_DIR = get_manifests_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()

MART_PATH = GOLD_DIR / GOLD_MART_FILENAME
DICT_PATH = FEATURE_DEFINITIONS_DIR / "feature_dictionary.csv"
LEAKAGE_PATH = DATA_QUALITY_REPORTS_DIR / "gold_leakage_guard_report.csv"

print("MODELING_MODE:", MODELING_MODE)
print("CLEAR_MODEL_OUTPUTS:", CLEAR_MODEL_OUTPUTS)
print("MART_PATH:", MART_PATH)
print("DICT_PATH:", DICT_PATH)


## Clean model outputs


In [ ]:
if CLEAR_MODEL_OUTPUTS:
    print("Cleaning model results...")
    clean_model_outputs(model_results_dir=MODEL_RESULTS_DIR, manifests_dir=MANIFESTS_DIR)
else:
    print("Skipping model cleanup (CLEAR_MODEL_OUTPUTS=False).")


## Load Gold mart and feature dictionary


In [ ]:
if not MART_PATH.exists():
    raise FileNotFoundError(
        f"Gold mart missing: {MART_PATH}. Run 03_gold_feature_engineering.ipynb first."
    )
if not DICT_PATH.is_file():
    raise FileNotFoundError(
        f"Feature dictionary missing: {DICT_PATH}. Run notebook 03 first."
    )

gold_df = read_parquet_if_exists(MART_PATH)
if gold_df is None or gold_df.empty:
    raise ValueError(f"Gold mart at {MART_PATH} is missing or empty.")

feature_dict = pd.read_csv(DICT_PATH)
leakage_report = pd.read_csv(LEAKAGE_PATH) if LEAKAGE_PATH.is_file() else None

print("Gold shape:", gold_df.shape)
print("Target rate:", gold_df[TARGET_COLUMN].mean())
validate_no_leakage(gold_df, feature_dict)
print("Leakage guard: OK")
if leakage_report is not None:
    blocked = leakage_report.loc[leakage_report["allowed_for_modeling"] == False]
    print(f"Leakage report: {len(blocked)} columns blocked from modeling")

feature_columns = get_model_feature_columns(feature_dict)
print(f"Selected model features ({len(feature_columns)}):")
print(feature_columns)


## Resolve train/validation/test splits


In [ ]:
splits, split_meta = resolve_modeling_splits(gold_df, mode=MODELING_MODE)
train_df = splits["train"]
val_df = splits["validation"]
test_df = splits["test"]

print("Split metadata:", split_meta)
for name, part in splits.items():
    if part.empty:
        print(f"WARNING: split '{name}' is empty")
    else:
        rate = part[TARGET_COLUMN].mean()
        seasons = sorted(part["session_year"].dropna().unique().tolist()) if "session_year" in part.columns else []
        print(f"{name}: n={len(part)}, points_finish rate={rate:.4f}, seasons={seasons}")

if MODELING_MODE == "smoke":
    print("SMOKE MODE: wiring verification only — metrics are NOT official MBA evidence.")


## Baselines


In [ ]:
X_train, y_train = prepare_model_matrix(train_df, feature_columns)
X_val, y_val = prepare_model_matrix(val_df, feature_columns)
X_test, y_test = prepare_model_matrix(test_df, feature_columns)

baseline_rows = []
cm_parts = []
error_parts = []

for split_name, eval_df, y_true in [
    ("validation", val_df, y_val),
    ("test", test_df, y_test),
]:
    if len(eval_df) == 0:
        continue
    rand_pred, rand_proba = random_baseline_predictions(
        y_true, positive_rate=float(y_train.mean()) if len(y_train) else None
    )
    baseline_rows.append(
        compute_classification_metrics(y_true, rand_pred, rand_proba, "random_baseline", split_name)
    )
    cm_parts.append(compute_confusion_matrix_table(y_true, rand_pred, "random_baseline", split_name))
    error_parts.append(
        build_error_analysis(eval_df, y_true, rand_pred, rand_proba, "random_baseline", split_name)
    )

    heur_pred, heur_proba = heuristic_position_baseline(eval_df)
    baseline_rows.append(
        compute_classification_metrics(y_true, heur_pred, heur_proba, "heuristic_position", split_name)
    )
    cm_parts.append(compute_confusion_matrix_table(y_true, heur_pred, "heuristic_position", split_name))
    error_parts.append(
        build_error_analysis(eval_df, y_true, heur_pred, heur_proba, "heuristic_position", split_name)
    )

baseline_metrics = pd.DataFrame(baseline_rows)
display(baseline_metrics)


## Train supervised models


In [ ]:
model_specs = {
    "logistic_regression": build_logistic_regression_pipeline(feature_columns, X_train),
    "random_forest": build_random_forest_pipeline(feature_columns, X_train),
    "lightgbm": build_lightgbm_pipeline(feature_columns, X_train),
}
fitted_models = train_models(X_train, y_train, model_specs)
list(fitted_models.keys())


## Validation and test evaluation


In [ ]:
val_rows = []
test_rows = []
importance_parts = []

for model_name, pipeline in fitted_models.items():
    if len(X_val) > 0:
        val_pred = pipeline.predict(X_val)
        val_proba = pipeline.predict_proba(X_val)[:, 1] if hasattr(pipeline, "predict_proba") else None
        val_rows.append(compute_classification_metrics(y_val, val_pred, val_proba, model_name, "validation"))
        cm_parts.append(compute_confusion_matrix_table(y_val, val_pred, model_name, "validation"))
        error_parts.append(build_error_analysis(val_df, y_val, val_pred, val_proba, model_name, "validation"))
    if len(X_test) > 0:
        test_pred = pipeline.predict(X_test)
        test_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None
        test_rows.append(compute_classification_metrics(y_test, test_pred, test_proba, model_name, "test"))
        cm_parts.append(compute_confusion_matrix_table(y_test, test_pred, model_name, "test"))
        error_parts.append(build_error_analysis(test_df, y_test, test_pred, test_proba, model_name, "test"))
    importance_parts.append(extract_feature_importance(model_name, pipeline, feature_columns))

validation_metrics = pd.DataFrame(val_rows)
test_metrics = pd.DataFrame(test_rows)
confusion_matrix_df = pd.concat(cm_parts, ignore_index=True) if cm_parts else pd.DataFrame()
error_analysis_df = pd.concat(error_parts, ignore_index=True) if error_parts else pd.DataFrame()
feature_importance_df = pd.concat(importance_parts, ignore_index=True) if importance_parts else pd.DataFrame()

display(validation_metrics)
display(test_metrics)


## Save modeling outputs


In [ ]:
output_paths = save_modeling_outputs(
    baseline_metrics=baseline_metrics,
    validation_metrics=validation_metrics,
    test_metrics=test_metrics,
    confusion_matrix_df=confusion_matrix_df,
    error_analysis_df=error_analysis_df,
    feature_importance_df=feature_importance_df,
    model_results_dir=MODEL_RESULTS_DIR,
    manifests_dir=MANIFESTS_DIR,
    manifest_extra={
        "modeling_mode": MODELING_MODE,
        "split_method": split_meta.get("split_method"),
        "evidence_tier": split_meta.get("evidence_tier"),
        "random_seed": RANDOM_SEED,
        "target": TARGET_COLUMN,
        "feature_count": len(feature_columns),
        "models": list(fitted_models.keys()),
    },
)
output_paths


## Next steps

- [ ] Review `reports/model_results/` CSVs on Drive
- [ ] For official MBA evidence: set `MODELING_MODE = "full"` after 2023–2025 Gold run
- [ ] Proceed to **05 — report artifacts** to build tables and figures
